# Ứng dụng 2 — Dự đoán giá nhà

**Assignment 02 — Phát triển các Hệ thống Thông minh**
Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · GVHD: PGS.TS Trần Đình Quế

Cùng một chuỗi xử lý với Ứng dụng 1, nhưng **loại bài toán khác** (hồi quy) và
**loại dữ liệu khác** (có biến phân loại). Chính hai khác biệt ấy là thứ Chương X
đem ra so sánh.

## 1. Định nghĩa bài toán

**Mục tiêu.** Ước lượng giá rao bán của một bất động sản nhà ở tại Việt Nam từ
các đặc điểm của chính bất động sản đó.

- $X$ = đặc điểm căn nhà (diện tích, số tầng, số phòng ngủ, hướng, tình trạng pháp lý, tỉnh/thành…)
- $y$ = giá nhà, tính bằng **tỷ VNĐ**, $y \in \mathbb{R}^{+}$

Đây là bài toán **hồi quy có giám sát**.

**Vì sao khác với Ứng dụng 1.** Ở tiểu đường, đầu ra là một trong hai nhãn rời
rạc và sai lầm được đếm bằng số ca. Ở đây đầu ra là **một số thực liên tục** và
sai lầm được đo bằng **khoảng cách**: đoán 5,0 tỷ cho căn 5,2 tỷ là gần đúng,
đoán 9,0 tỷ là sai nặng. Hệ quả trực tiếp: không có ma trận nhầm lẫn, không có
accuracy, không có ROC-AUC. Bộ độ đo phải đổi hoàn toàn sang MAE, MSE, RMSE, $R^2$.

## 2. Nguồn dữ liệu

| Mục | Giá trị |
|---|---|
| Tên tập dữ liệu | Vietnam Housing Dataset 2024 |
| Nguồn Kaggle | https://www.kaggle.com/datasets/nguyentiennhan/vietnam-housing-dataset-2024 |
| Số quan sát | 30 229 |
| Số thuộc tính | 12 (11 đặc trưng + 1 biến mục tiêu) |
| Biến mục tiêu | `Price` (tỷ VNĐ) |
| Tệp cục bộ | `data/house_prices.csv` |

**Một quan sát là gì.** Mỗi dòng là **một tin rao bán bất động sản** trên sàn
giao dịch trực tuyến tại Việt Nam năm 2024. Đây là một điểm cần nói rõ:
`Price` là **giá chào bán**, không phải giá giao dịch thành công. Mô hình vì vậy
học "thị trường đang hỏi bao nhiêu", chứ không phải "thị trường trả bao nhiêu" —
một giới hạn thật, sẽ được nhắc lại ở phần kết luận.

In [1]:
# --- 3. Nạp dữ liệu ---
import json
import random
import re
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["font.family"] = "DejaVu Sans"

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "house_prices.csv"
MODEL_DIR = ROOT / "model"
FIG_DIR = ROOT.parent / "report" / "figures"
MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)
df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
print("Đã nạp:", DATA)
print("Kích thước (N, cột):", df.shape)
df.head()

Đã nạp: D:\Python\HTTM_Assignment02\Assignment-02-Intelligent-System\house_price\data\house_prices.csv
Kích thước (N, cột): (30229, 12)


,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price
0,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",84.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,Have certificate,NaN,8.60
1,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",60.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,7.50
2,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,NaN,NaN,Sale contract,NaN,8.90
3,"Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...",54.0,NaN,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,Full,5.35
4,"Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh",92.0,NaN,NaN,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,Full,6.90


## 4. Khảo sát cấu trúc dữ liệu

In [2]:
print("--- df.shape ---"); print(df.shape)
print("\n--- df.info() ---"); df.info()
print("\n--- df.describe() (cột số) ---")
display(df.describe().T.round(3))

--- df.shape ---
(30229, 12)

--- df.info() ---
<class 'pandas.DataFrame'>
RangeIndex: 30229 entries, 0 to 30228
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Address            30229 non-null  str    
 1   Area               30229 non-null  float64
 2   Frontage           18665 non-null  float64
 3   Access Road        16932 non-null  float64
 4   House direction    8990 non-null   str    
 5   Balcony direction  5246 non-null   str    
 6   Floors             26626 non-null  float64
 7   Bedrooms           25067 non-null  float64
 8   Bathrooms          23155 non-null  float64
 9   Legal status       25723 non-null  str    
 10  Furniture state    16110 non-null  str    
 11  Price              30229 non-null  float64
dtypes: float64(7), str(5)
memory usage: 5.4 MB

--- df.describe() (cột số) ---


,count,mean,std,min,25%,50%,75%,max
Area,30229.0,68.499,48.070,3.1,40.0,56.0,80.0,595.0
Frontage,18665.0,5.362,4.346,1.0,4.0,4.5,5.0,77.0
Access Road,16932.0,7.854,7.451,1.0,4.0,6.0,10.0,85.0
Floors,26626.0,3.410,1.329,1.0,2.0,3.0,4.0,10.0
Bedrooms,25067.0,3.511,1.309,1.0,3.0,3.0,4.0,9.0
Bathrooms,23155.0,3.347,1.400,1.0,2.0,3.0,4.0,9.0
Price,30229.0,5.872,2.212,1.0,4.2,5.9,7.5,11.5


In [3]:
print("--- Giá trị thiếu theo cột ---")
miss = pd.DataFrame({"so_thieu": df.isna().sum()})
miss["ty_le_%"] = (miss["so_thieu"] / len(df) * 100).round(2)
display(miss.sort_values("ty_le_%", ascending=False))

print("\n--- Số bản ghi trùng lặp hoàn toàn ---")
print(int(df.duplicated().sum()))

--- Giá trị thiếu theo cột ---


,so_thieu,ty_le_%
Balcony direction,24983,82.65
House direction,21239,70.26
Furniture state,14119,46.71
Access Road,13297,43.99
Frontage,11564,38.25
Bathrooms,7074,23.40
Bedrooms,5162,17.08
Legal status,4506,14.91
Floors,3603,11.92
Area,0,0.00



--- Số bản ghi trùng lặp hoàn toàn ---
0


### Phân loại cột theo vai trò

| Nhóm | Cột |
|---|---|
| **Số (numerical)** | `Area`, `Frontage`, `Access Road`, `Floors`, `Bedrooms`, `Bathrooms` |
| **Phân loại (categorical)** | `House direction`, `Balcony direction`, `Legal status`, `Furniture state` |
| **Văn bản (text)** | `Address` — địa chỉ tự do, sẽ được rút thành tỉnh/thành |
| **Mục tiêu (target)** | `Price` |

Khác hẳn Ứng dụng 1: ở đây **có bốn biến phân loại**, nên bước mã hoá one-hot
là bắt buộc. Đây chính là nội dung mục 3.5 của đề bài
($\{\text{Urban}, \text{Suburban}, \text{Rural}\} \rightarrow$ vectơ nhị phân).

## 5–9. Phân tích chất lượng dữ liệu

Bốn vấn đề cần kiểm: giá trị thiếu, trùng lặp, giá trị không hợp lệ, và ngoại lệ.

In [4]:
fig, ax = plt.subplots(figsize=(8, 4.4))
m = miss[miss["ty_le_%"] > 0].sort_values("ty_le_%")
ax.barh(m.index, m["ty_le_%"], color="#c0392b")
for i, v in enumerate(m["ty_le_%"]):
    ax.text(v + 1, i, f"{v}%", va="center", fontsize=9)
ax.set_xlabel("Tỷ lệ giá trị thiếu (%)")
ax.set_title("Ứng dụng 2 — Tỷ lệ giá trị thiếu theo cột")
ax.set_xlim(0, 100)
fig.savefig(FIG_DIR / "hou_missing.png")
plt.show()

**Quan sát.** `Balcony direction` thiếu 82,6%, `House direction` thiếu 70,3%,
`Access Road` 44,0%, `Furniture state` 46,7%, `Frontage` 38,3%. `Area` và `Price`
không thiếu dòng nào.

**Diễn giải.** Đây **không phải lỗi thu thập** mà là **hành vi người đăng tin**:
biểu mẫu đăng tin chỉ bắt buộc diện tích và giá; các trường còn lại tuỳ chọn nên
người bán chỉ điền khi thấy có lợi. Nghĩa là bản thân việc **thiếu** cũng mang
thông tin (tin đăng sơ sài thường là môi giới đăng hàng loạt).

**Ý nghĩa với học máy.** Không thể xoá dòng theo `Balcony direction` — sẽ mất
82,6% dữ liệu. Cũng không nên xoá cột ngay, vì với biến phân loại ta có một lựa
chọn sạch hơn: coi **"không khai báo" là một hạng mục riêng** (`Không rõ`) rồi
one-hot nó. Cách này giữ nguyên số dòng và biến sự vắng mặt thành tín hiệu.
Với cột số thì điền bằng **trung vị** của tập train (mục 15).

In [5]:
# --- 8. Giá trị không hợp lệ ---
invalid = {
    "Area <= 0": int((df["Area"] <= 0).sum()),
    "Price <= 0": int((df["Price"] <= 0).sum()),
    "Bedrooms > 20": int((df["Bedrooms"] > 20).sum()),
    "Bathrooms > 20": int((df["Bathrooms"] > 20).sum()),
    "Floors > 20": int((df["Floors"] > 20).sum()),
    "Area > 10000 m2": int((df["Area"] > 10000).sum()),
}
display(pd.Series(invalid, name="so_ban_ghi").to_frame())

print("Số bản ghi trùng lặp hoàn toàn:", int(df.duplicated().sum()))
print("Số bản ghi trùng trên (Address, Area, Price):",
      int(df.duplicated(subset=["Address", "Area", "Price"]).sum()))

,so_ban_ghi
Area <= 0,0
Price <= 0,0
Bedrooms > 20,0
Bathrooms > 20,0
Floors > 20,0
Area > 10000 m2,0


Số bản ghi trùng lặp hoàn toàn: 0
Số bản ghi trùng trên (Address, Area, Price): 2699


**Quan sát.** Có bản ghi trùng lặp — cùng địa chỉ, cùng diện tích, cùng giá.

**Diễn giải.** Một bất động sản thường được đăng lại nhiều lần, hoặc do nhiều
môi giới cùng đăng một căn.

**Ý nghĩa với học máy.** Phải khử trùng **trước khi chia train/test**. Nếu không,
cùng một căn nhà có thể xuất hiện ở cả hai tập; mô hình được chấm điểm trên đúng
căn nó đã học thuộc, và $R^2$ trên test bị thổi phồng. Đây là dạng rò rỉ dữ liệu
tinh vi nhất trong ba ứng dụng vì nó không lộ ra ở bất kỳ biểu đồ nào.

In [6]:
# --- 9. Ngoại lệ theo IQR ---
NUM_RAW = ["Area", "Frontage", "Access Road", "Floors", "Bedrooms", "Bathrooms", "Price"]
rows = []
for c in NUM_RAW:
    s = df[c].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = int(((s < lo) | (s > hi)).sum())
    rows.append({"cot": c, "min": round(s.min(), 2), "Q1": round(q1, 2),
                 "trung_vi": round(s.median(), 2), "Q3": round(q3, 2),
                 "max": round(s.max(), 2), "so_ngoai_le": n,
                 "ty_le_%": round(n / len(s) * 100, 2)})
display(pd.DataFrame(rows).set_index("cot"))

fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
for ax, c in zip(axes.ravel(), NUM_RAW):
    sns.boxplot(y=df[c].dropna(), ax=ax, color="#2980b9", width=0.45)
    ax.set_title(c, fontsize=10); ax.set_ylabel("")
axes.ravel()[-1].axis("off")
fig.suptitle("Ứng dụng 2 — Hộp râu các biến số", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "hou_boxplot.png")
plt.show()

,min,Q1,trung_vi,Q3,max,so_ngoai_le,ty_le_%
cot,,,,,,,
Area,3.1,40.0,56.0,80.0,595.0,1636,5.41
Frontage,1.0,4.0,4.5,5.0,77.0,2305,12.35
Access Road,1.0,4.0,6.0,10.0,85.0,1110,6.56
Floors,1.0,2.0,3.0,4.0,10.0,5,0.02
Bedrooms,1.0,3.0,3.0,4.0,9.0,2532,10.10
Bathrooms,1.0,2.0,3.0,4.0,9.0,287,1.24
Price,1.0,4.2,5.9,7.5,11.5,0,0.00


**Quan sát.** `Area` có đuôi phải cực dài (tối đa hàng nghìn m²) trong khi trung
vị chỉ vài chục m². `Price` bị chặn ở 11,5 tỷ.

**Diễn giải.** Đuôi `Area` là các lô đất lớn hoặc nhà xưởng lẫn vào tập dữ liệu
nhà ở. Việc `Price` chặn ở 11,5 tỷ cho thấy **tập dữ liệu đã được lọc sẵn** ở
khoảng phổ thông — đây là điều phải nói rõ chứ không được bỏ qua, vì nó giới hạn
phạm vi áp dụng của mô hình.

**Ý nghĩa với học máy.** Ta **cắt ngưỡng có kiểm soát** ở mục 12: giữ các căn có
diện tích trong khoảng hợp lý cho nhà ở đô thị. Đây không phải "xoá dữ liệu xấu"
mà là **thu hẹp phạm vi bài toán một cách minh bạch** — mô hình dự đoán nhà ở đô
thị, và ta khai báo đúng như vậy.

## 10. Phân tích khám phá dữ liệu (EDA)

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
sns.histplot(df["Price"], bins=60, kde=True, ax=axes[0], color="#8e44ad")
axes[0].set_title("Phân phối giá nhà")
axes[0].set_xlabel("Giá (tỷ VNĐ)"); axes[0].set_ylabel("Số tin đăng")
axes[0].axvline(df["Price"].median(), color="red", ls="--",
                label=f"Trung vị = {df['Price'].median():.2f} tỷ")
axes[0].legend(fontsize=9)
sns.histplot(np.log1p(df["Price"]), bins=60, kde=True, ax=axes[1], color="#16a085")
axes[1].set_title("Phân phối log(1 + giá)")
axes[1].set_xlabel("log(1 + giá)"); axes[1].set_ylabel("")
fig.suptitle("Ứng dụng 2 — Phân phối biến mục tiêu", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "hou_target.png")
plt.show()
print(df["Price"].describe().round(3))
print("Độ lệch (skewness):", round(float(df["Price"].skew()), 3))

count    30229.000
mean         5.872
std          2.212
min          1.000
25%          4.200
50%          5.900
75%          7.500
max         11.500
Name: Price, dtype: float64
Độ lệch (skewness): -0.029


**Quan sát.** Giá tập trung quanh trung vị 5,9 tỷ, phân phối gần đối xứng, độ
lệch nhỏ. Không có đuôi phải cực đoan như thường thấy ở dữ liệu bất động sản.

**Diễn giải.** Do tập dữ liệu đã bị chặn trên ở 11,5 tỷ (mục 9). Ở một tập không
bị chặn, giá bất động sản luôn lệch phải mạnh và ta sẽ phải huấn luyện trên
$\log(1 + y)$ rồi mũ hoá ngược khi dự đoán.

**Ý nghĩa với học máy.** Vì phân phối đã gần đối xứng, ta **huấn luyện thẳng trên
$y$** mà không cần biến đổi log. Quyết định này giữ cho MAE và RMSE đọc được trực
tiếp bằng tỷ VNĐ — người dùng cuối hiểu ngay "sai trung bình 1,2 tỷ" mà không cần
quy đổi.

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
sub = df[(df.Area > 0) & (df.Area < 500)]
axes[0].scatter(sub["Area"], sub["Price"], s=5, alpha=0.18, color="#2980b9")
axes[0].set_xlabel("Diện tích (m²)"); axes[0].set_ylabel("Giá (tỷ VNĐ)")
axes[0].set_title("Giá theo diện tích")

bed = df[df.Bedrooms.between(1, 8)]
sns.boxplot(x="Bedrooms", y="Price", data=bed, ax=axes[1], palette="viridis")
axes[1].set_title("Giá theo số phòng ngủ")
axes[1].set_xlabel("Số phòng ngủ"); axes[1].set_ylabel("")

flo = df[df.Floors.between(1, 7)]
sns.boxplot(x="Floors", y="Price", data=flo, ax=axes[2], palette="magma")
axes[2].set_title("Giá theo số tầng")
axes[2].set_xlabel("Số tầng"); axes[2].set_ylabel("")
fig.suptitle("Ứng dụng 2 — Quan hệ giữa đặc trưng và giá", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "hou_relations.png")
plt.show()

**Quan sát.** Giá tăng theo diện tích nhưng **độ tán rất lớn**: hai căn cùng 80 m²
có thể chênh nhau vài tỷ. Giá tăng đều theo số phòng ngủ và số tầng.

**Diễn giải.** Diện tích một mình không quyết định giá — **vị trí** mới quyết
định, và vị trí đang nằm ẩn trong cột `Address` dạng văn bản tự do mà mô hình
chưa đọc được.

**Ý nghĩa với học máy.** Đây là căn cứ trực tiếp cho kỹ thuật đặc trưng ở mục 13:
phải **rút tỉnh/thành ra khỏi `Address`** và mã hoá thành biến phân loại. Nếu bỏ
qua bước này, mô hình sẽ mãi không giải thích được phần lớn phương sai của giá.

In [9]:
fig, ax = plt.subplots(figsize=(7.6, 6))
corr = df[NUM_RAW].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Ứng dụng 2 — Ma trận tương quan Pearson")
fig.savefig(FIG_DIR / "hou_corr.png")
plt.show()
print("Tương quan với Price:")
print(corr["Price"].drop("Price").sort_values(ascending=False).round(3))

Tương quan với Price:
Bathrooms      0.434
Bedrooms       0.389
Floors         0.332
Access Road    0.164
Area           0.098
Frontage       0.047
Name: Price, dtype: float64


**Quan sát.** Tương quan tuyến tính giữa từng biến số và `Price` đều **thấp**
(dưới 0,3). Cặp `Bedrooms`–`Bathrooms` tương quan với nhau khá cao.

**Diễn giải.** Tương quan Pearson chỉ đo quan hệ **tuyến tính**. Giá bất động sản
phụ thuộc vị trí theo cách phi tuyến và theo tương tác (80 m² ở Hà Nội khác hẳn
80 m² ở tỉnh lẻ), nên hệ số tuyến tính thấp là điều được dự đoán trước.

**Ý nghĩa với học máy.** Đây là dự báo mạnh rằng **Linear Regression sẽ kém hơn
các mô hình dạng cây** ở bài toán này — và bảng kết quả mục 19 sẽ xác nhận. Ở
Ứng dụng 1 thì ngược lại: `Glucose` có tương quan tuyến tính 0,49 với nhãn nên
Logistic Regression cạnh tranh tốt.

## 11–13. Kiểu đặc trưng, biểu diễn và kỹ thuật đặc trưng

**Kỹ thuật đặc trưng chính: rút tỉnh/thành từ `Address`.**

`Address` là chuỗi tự do dạng
`"Dự án ..., Xã Long Hưng, Văn Giang, Hưng Yên"`. Thành phần cuối cùng sau dấu
phẩy là **tỉnh/thành phố** — đây chính là biến vị trí mà mục 10 chỉ ra là thiếu.

Ta **không** one-hot toàn bộ chuỗi địa chỉ (mỗi địa chỉ gần như duy nhất → hàng
chục nghìn cột, mô hình học thuộc lòng chứ không khái quát). Thay vào đó rút ra
tỉnh/thành, giữ những tỉnh có đủ số lượng tin đăng, gộp phần còn lại thành
`Khác`. Đây là đánh đổi có chủ ý: **mất độ chi tiết (quận/phường), giữ được khả
năng khái quát**.

In [10]:
def parse_address(address):
    '''Rút (tỉnh/thành, quận/huyện) từ chuỗi địa chỉ tự do.

    Địa chỉ hành chính Việt Nam viết từ nhỏ đến lớn:
    "... , Phường/Xã , Quận/Huyện , Tỉnh/Thành". Vì vậy thành phần cuối là cấp
    tỉnh và áp chót là cấp huyện. Chuỗi dưới 3 đoạn là tiêu đề tin rao chứ không
    phải địa chỉ (ví dụ "Bán nhà chính chủ Phó Đức Chính") nên không tách được.
    '''
    if not isinstance(address, str) or not address.strip():
        return "Không rõ", "Không rõ"
    parts = [p.strip().rstrip(".").strip() for p in address.split(",") if p.strip()]
    if len(parts) < 3:
        return "Không rõ", "Không rõ"
    return (parts[-1] or "Không rõ"), (parts[-2] or "Không rõ")


parsed = df["Address"].apply(parse_address)
df["Province"] = parsed.str[0]
df["District"] = parsed.str[1]
prov_counts = df["Province"].value_counts()
print("Số tỉnh/thành khác nhau rút được:", df["Province"].nunique())
print("\n15 tỉnh/thành nhiều tin đăng nhất:")
print(prov_counts.head(15))

MIN_PROV = 150
KEEP_PROVINCES = sorted(prov_counts[prov_counts >= MIN_PROV].index.tolist())
print(f"\nGiữ {len(KEEP_PROVINCES)} tỉnh/thành có >= {MIN_PROV} tin đăng; phần còn lại gộp thành 'Khác'.")
df["Province"] = df["Province"].where(df["Province"].isin(KEEP_PROVINCES), "Khác")
print("Số hạng mục Province sau khi gộp:", df["Province"].nunique())

# Cấp huyện cho vị trí chi tiết hơn cấp tỉnh, nhưng đuôi dài hơn nhiều nên
# ngưỡng gộp phải cao hơn để tránh sinh ra hàng trăm cột one-hot học thuộc lòng.
MIN_DIST = 80
dist_counts = df["District"].value_counts()
KEEP_DISTRICTS = sorted(dist_counts[dist_counts >= MIN_DIST].index.tolist())
df["District"] = df["District"].where(df["District"].isin(KEEP_DISTRICTS), "Khác")
print(f"Số quận/huyện khác nhau rút được: {dist_counts.size}")
print(f"Giữ {len(KEEP_DISTRICTS)} quận/huyện có >= {MIN_DIST} tin đăng → "
      f"{df['District'].nunique()} hạng mục sau khi gộp")

Số tỉnh/thành khác nhau rút được: 76

15 tỉnh/thành nhiều tin đăng nhất:
Province
Hồ Chí Minh        11778
Hà Nội             10457
Bình Dương          1671
Đà Nẵng             1437
Đồng Nai             829
Hải Phòng            789
Khánh Hòa            743
Hưng Yên             404
Long An              339
Bà Rịa Vũng Tàu      240
Bắc Ninh             167
Bình Thuận           127
Lâm Đồng             118
Cần Thơ              111
Quảng Ninh           110
Name: count, dtype: int64

Giữ 11 tỉnh/thành có >= 150 tin đăng; phần còn lại gộp thành 'Khác'.
Số hạng mục Province sau khi gộp: 12
Số quận/huyện khác nhau rút được: 343
Giữ 63 quận/huyện có >= 80 tin đăng → 64 hạng mục sau khi gộp


In [11]:
fig, ax = plt.subplots(figsize=(9.5, 5))
top = df.groupby("Province")["Price"].agg(["median", "count"])
top = top[top["count"] >= 200].sort_values("median").tail(18)
ax.barh(top.index, top["median"], color="#e67e22")
for i, (v, n) in enumerate(zip(top["median"], top["count"])):
    ax.text(v + 0.05, i, f"{v:.1f} tỷ (n={n})", va="center", fontsize=8)
ax.set_xlabel("Giá trung vị (tỷ VNĐ)")
ax.set_title("Ứng dụng 2 — Giá trung vị theo tỉnh/thành phố")
fig.savefig(FIG_DIR / "hou_province.png")
plt.show()

**Quan sát.** Giá trung vị chênh lệch rõ rệt giữa các tỉnh/thành.

**Diễn giải.** Xác nhận giả thuyết ở mục 10: vị trí là biến giải thích mạnh, và
nó vốn bị chôn trong một cột văn bản.

**Ý nghĩa với học máy.** Chỉ một phép rút chuỗi đơn giản đã biến dữ liệu văn bản
không dùng được thành đặc trưng phân loại có sức dự báo. Đây là minh hoạ trực
tiếp cho luận điểm của Assignment 02: **biểu diễn dữ liệu là một phần của lời
giải, không phải công đoạn phụ trợ**.

### Năm đặc trưng phái sinh

Ngoài `Province`, ta tạo thêm năm đặc trưng không có sẵn trong CSV. Mỗi cái mã
hoá một quan hệ mà mô hình tuyến tính không tự tìm ra được từ các cột gốc:

| Đặc trưng | Công thức | Mã hoá điều gì |
|---|---|---|
| `Total_Area` | $\text{Area} \times \max(\text{Floors}, 1)$ | Tổng diện tích sàn sử dụng — thứ người mua thực sự trả tiền, khác hẳn diện tích đất |
| `Room_Density` | $(\text{Bedrooms} + \text{Bathrooms}) / \text{Area}$ | Mức độ chia nhỏ. Mật độ cao gợi ý nhà trọ cho thuê chứ không phải nhà ở gia đình |
| `Frontage_Ratio` | $\text{Frontage} / \text{Area}$ | Hình dạng lô đất. Tỷ lệ thấp là nhà ống sâu, tỷ lệ cao là lô vuông vắn — hai phân khúc giá khác nhau |
| `Area_per_Bedroom` | $\text{Area} / \max(\text{Bedrooms}, 1)$ | Độ rộng rãi mỗi phòng, phân biệt nhà cao cấp với nhà chia nhỏ |
| `Log_Area` | $\log(1 + \text{Area})$ | Nén đuôi phải của diện tích, giúp mô hình tuyến tính bắt được quan hệ dưới tuyến tính giữa diện tích và giá |

**Vì sao ba tỷ số lại có ích với mô hình dạng cây.** Cây quyết định chia theo
**một biến mỗi lần**, nên nó không thể biểu diễn `Bedrooms / Area` trừ khi được
cho sẵn — nó sẽ phải xấp xỉ bằng rất nhiều nhát cắt bậc thang trên hai biến riêng
lẻ, tốn độ sâu và dễ quá khớp. Đưa sẵn tỷ số vào là cách **giảm gánh nặng biểu
diễn** cho mô hình. Mục 19 sẽ đo hiệu quả thật của quyết định này.

In [12]:
# --- Làm sạch có kiểm soát ---
before = len(df)
clean = df.drop_duplicates(subset=["Address", "Area", "Price"]).copy()
after_dup = len(clean)

clean = clean[(clean["Area"] > 10) & (clean["Area"] <= 1000)]
clean = clean[(clean["Price"] > 0)]
clean = clean[(clean["Bedrooms"].isna()) | (clean["Bedrooms"] <= 15)]
clean = clean[(clean["Bathrooms"].isna()) | (clean["Bathrooms"] <= 15)]
clean = clean[(clean["Floors"].isna()) | (clean["Floors"] <= 15)]
after = len(clean)

print(f"Ban đầu                    : {before:>6} dòng")
print(f"Sau khi khử trùng lặp      : {after_dup:>6} dòng  (-{before-after_dup})")
print(f"Sau khi lọc giá trị bất hợp lý: {after:>6} dòng  (-{after_dup-after})")
print(f"Giữ lại                    : {after/before*100:.1f}% dữ liệu gốc")

# --- Kỹ thuật đặc trưng phái sinh (cải tiến so với Assignment 01) ---
clean["Total_Area"] = clean["Area"] * clean["Floors"].fillna(1).clip(lower=1)
clean["Room_Density"] = ((clean["Bedrooms"].fillna(0) + clean["Bathrooms"].fillna(0))
                         / clean["Area"].clip(lower=1))
clean["Frontage_Ratio"] = clean["Frontage"] / clean["Area"].clip(lower=1)
clean["Area_per_Bedroom"] = clean["Area"] / clean["Bedrooms"].fillna(1).clip(lower=1)
clean["Log_Area"] = np.log1p(clean["Area"])

CAT_FEATURES = ["House direction", "Balcony direction", "Legal status",
                "Furniture state", "Province", "District"]
NUM_BASE = ["Area", "Frontage", "Access Road", "Floors", "Bedrooms", "Bathrooms"]
NUM_DERIVED = ["Total_Area", "Room_Density", "Frontage_Ratio", "Area_per_Bedroom", "Log_Area"]
NUM_FEATURES = NUM_BASE + NUM_DERIVED
FEATURES = NUM_FEATURES + CAT_FEATURES

print("")
print("Đặc trưng phái sinh được thêm (không có trong dữ liệu gốc):")
for c in NUM_DERIVED:
    print(f"   {c:<18} min={clean[c].min():>10.3f}  trung vị={clean[c].median():>10.3f}  max={clean[c].max():>12.3f}")

for c in CAT_FEATURES:
    clean[c] = clean[c].fillna("Không rõ").astype(str).str.strip()
    clean.loc[clean[c] == "", c] = "Không rõ"

print("\nSố hạng mục của từng biến phân loại (đã gộp 'Không rõ'):")
for c in CAT_FEATURES:
    print(f"   {c:<20} {clean[c].nunique():>3} hạng mục")

Ban đầu                    :  30229 dòng
Sau khi khử trùng lặp      :  27530 dòng  (-2699)
Sau khi lọc giá trị bất hợp lý:  27513 dòng  (-17)
Giữ lại                    : 91.0% dữ liệu gốc

Đặc trưng phái sinh được thêm (không có trong dữ liệu gốc):
   Total_Area         min=    10.350  trung vị=   170.000  max=    1760.000
   Room_Density       min=     0.000  trung vị=     0.096  max=       0.583
   Frontage_Ratio     min=     0.006  trung vị=     0.078  max=       2.026
   Area_per_Bedroom   min=     3.750  trung vị=    19.200  max=     577.600
   Log_Area           min=     2.429  trung vị=     4.043  max=       6.390

Số hạng mục của từng biến phân loại (đã gộp 'Không rõ'):
   House direction        9 hạng mục
   Balcony direction      9 hạng mục
   Legal status           3 hạng mục
   Furniture state        3 hạng mục
   Province              12 hạng mục
   District              64 hạng mục


### Biểu diễn dữ liệu — từ CSV đến ma trận đặc trưng

$$\text{CSV thô} \rightarrow \text{DataFrame} \rightarrow \text{dữ liệu sạch}
\rightarrow \text{đặc trưng đã mã hoá và chuẩn hoá} \rightarrow X$$

Ô dưới in ra một bản ghi CSV gốc, các thành phần của nó, và hình dạng ma trận.

In [13]:
print("=" * 76)
print("BƯỚC 1 — MỘT BẢN GHI CSV GỐC")
print("=" * 76)
with open(DATA, encoding="utf-8-sig") as f:
    print("Tiêu đề  :", f.readline().strip())
    print("Dữ liệu  :", f.readline().strip())

print("\n" + "=" * 76)
print("BƯỚC 2 — BẢN GHI ẤY TRONG DATAFRAME (đã thêm cột Province rút ra)")
print("=" * 76)
display(clean.head(1)[FEATURES + ["Price"]])

print("=" * 76)
print("BƯỚC 3 — TÁCH THÀNH PHẦN SỐ VÀ THÀNH PHẦN PHÂN LOẠI")
print("=" * 76)
r0 = clean.iloc[0]
print("Thành phần SỐ (giữ nguyên, chỉ điền khuyết + chuẩn hoá):")
for c in NUM_FEATURES:
    print(f"   {c:<16} = {r0[c]}")
print("\nThành phần PHÂN LOẠI (sẽ được one-hot):")
for c in CAT_FEATURES:
    print(f"   {c:<20} = '{r0[c]}'")
print(f"\n   y = {r0['Price']} tỷ VNĐ")

BƯỚC 1 — MỘT BẢN GHI CSV GỐC
Tiêu đề  : Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price
Dữ liệu  : "Dự án The Empire - Vinhomes Ocean Park 2, Xã Long Hưng, Văn Giang, Hưng Yên",84,,,,,4,,,Have certificate,,8.6

BƯỚC 2 — BẢN GHI ẤY TRONG DATAFRAME (đã thêm cột Province rút ra)


,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,Total_Area,Room_Density,Frontage_Ratio,Area_per_Bedroom,Log_Area,House direction,Balcony direction,Legal status,Furniture state,Province,District,Price
0,84.0,NaN,NaN,4.0,NaN,NaN,336.0,0.0,NaN,84.0,4.442651,Không rõ,Không rõ,Have certificate,Không rõ,Hưng Yên,Văn Giang,8.6


BƯỚC 3 — TÁCH THÀNH PHẦN SỐ VÀ THÀNH PHẦN PHÂN LOẠI
Thành phần SỐ (giữ nguyên, chỉ điền khuyết + chuẩn hoá):
   Area             = 84.0
   Frontage         = nan
   Access Road      = nan
   Floors           = 4.0
   Bedrooms         = nan
   Bathrooms        = nan
   Total_Area       = 336.0
   Room_Density     = 0.0
   Frontage_Ratio   = nan
   Area_per_Bedroom = 84.0
   Log_Area         = 4.442651256490317

Thành phần PHÂN LOẠI (sẽ được one-hot):
   House direction      = 'Không rõ'
   Balcony direction    = 'Không rõ'
   Legal status         = 'Have certificate'
   Furniture state      = 'Không rõ'
   Province             = 'Hưng Yên'
   District             = 'Văn Giang'

   y = 8.6 tỷ VNĐ


### Mã hoá biến phân loại — one-hot

Đề bài yêu cầu giải thích cách $\{\text{Urban}, \text{Suburban}, \text{Rural}\}$
trở thành số. Ta dùng **one-hot encoding**: mỗi hạng mục thành một cột nhị phân.

Ví dụ với `Legal status` có 3 hạng mục:

$$\texttt{Have certificate} \rightarrow [1, 0, 0], \quad
\texttt{Sale contract} \rightarrow [0, 1, 0], \quad
\texttt{Không rõ} \rightarrow [0, 0, 1]$$

**Vì sao không dùng label encoding** (gán 0, 1, 2)? Vì nó áp đặt một **thứ tự**
không có thật lên dữ liệu: mô hình sẽ hiểu rằng `Sale contract` (1) nằm "giữa"
`Have certificate` (0) và `Không rõ` (2), và rằng khoảng cách từ 0 đến 2 gấp đôi
khoảng cách từ 0 đến 1. Với hướng nhà thì còn vô lý hơn — "Đông" không lớn hơn
"Tây". One-hot không giả định gì về thứ tự, đổi lại làm tăng số chiều.

## 14. Chia tập Train / Validation / Test

Tỷ lệ 70 / 15 / 15. **Không phân tầng** — đây là bài toán hồi quy, biến mục tiêu
liên tục nên không có "lớp" để giữ tỷ lệ. Với $N > 25\,000$, phép chia ngẫu nhiên
đủ để ba tập có phân phối giá tương đương; ô dưới kiểm chứng điều đó.

In [14]:
from sklearn.model_selection import train_test_split

X = clean[FEATURES].copy()
y = clean["Price"].copy()

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.1765, random_state=RANDOM_SEED)

display(pd.DataFrame({
    "Tập": ["Train", "Validation", "Test", "Tổng"],
    "Số mẫu": [len(X_train), len(X_val), len(X_test), len(X)],
    "Tỷ lệ %": [round(len(t)/len(X)*100, 1) for t in (X_train, X_val, X_test)] + [100.0],
    "Giá trung bình (tỷ)": [round(t.mean(), 3) for t in (y_train, y_val, y_test)] + [round(y.mean(), 3)],
    "Giá trung vị (tỷ)": [round(t.median(), 3) for t in (y_train, y_val, y_test)] + [round(y.median(), 3)],
}))
print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)

,Tập,Số mẫu,Tỷ lệ %,Giá trung bình (tỷ),Giá trung vị (tỷ)
0,Train,19258,70.0,5.857,5.85
1,Validation,4128,15.0,5.844,5.85
2,Test,4127,15.0,5.873,5.90
3,Tổng,27513,100.0,5.858,5.85


X_train: (19258, 17) | X_val: (4128, 17) | X_test: (4127, 17)


## 15. Pipeline tiền xử lý

Khác Ứng dụng 1, ở đây cần **hai nhánh xử lý song song** vì có hai loại cột.
`ColumnTransformer` là công cụ đúng cho việc ấy:

| Nhánh | Cột | Các bước |
|---|---|---|
| **Số** | 6 cột | `SimpleImputer(median)` → `StandardScaler` |
| **Phân loại** | 5 cột | `SimpleImputer(constant="Không rõ")` → `OneHotEncoder(handle_unknown="ignore")` |

`handle_unknown="ignore"` là chi tiết quan trọng cho khâu triển khai: nếu người
dùng Web nhập một tỉnh không có trong tập huấn luyện, encoder trả về vectơ toàn 0
thay vì **ném lỗi và làm sập API**.

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_branch = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_branch = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Không rõ")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_branch, NUM_FEATURES),
    ("cat", categorical_branch, CAT_FEATURES),
], remainder="drop")

preprocessor.fit(X_train)          # CHỈ fit trên train
Xtr = preprocessor.transform(X_train)
Xva = preprocessor.transform(X_val)
Xte = preprocessor.transform(X_test)

feat_names = preprocessor.get_feature_names_out()
print("Số chiều TRƯỚC mã hoá :", X_train.shape[1], "cột "
      f"({len(NUM_FEATURES)} số + {len(CAT_FEATURES)} phân loại)")
print("Số chiều SAU mã hoá   :", Xtr.shape[1], "cột")
print(f"   → one-hot làm tăng {Xtr.shape[1] - len(NUM_FEATURES)} cột từ "
      f"{len(CAT_FEATURES)} biến phân loại")
print("\nHình dạng đầu vào mô hình:")
print("   Xtr:", Xtr.shape, "| Xva:", Xva.shape, "| Xte:", Xte.shape)
print("   dtype:", Xtr.dtype)
print("\n20 tên cột đầu tiên sau biến đổi:")
print(list(feat_names[:20]))

Số chiều TRƯỚC mã hoá : 17 cột (11 số + 6 phân loại)
Số chiều SAU mã hoá   : 111 cột
   → one-hot làm tăng 100 cột từ 6 biến phân loại

Hình dạng đầu vào mô hình:
   Xtr: (19258, 111) | Xva: (4128, 111) | Xte: (4127, 111)
   dtype: float64

20 tên cột đầu tiên sau biến đổi:
['num__Area', 'num__Frontage', 'num__Access Road', 'num__Floors', 'num__Bedrooms', 'num__Bathrooms', 'num__Total_Area', 'num__Room_Density', 'num__Frontage_Ratio', 'num__Area_per_Bedroom', 'num__Log_Area', 'cat__House direction_Bắc', 'cat__House direction_Không rõ', 'cat__House direction_Nam', 'cat__House direction_Tây', 'cat__House direction_Tây - Bắc', 'cat__House direction_Tây - Nam', 'cat__House direction_Đông', 'cat__House direction_Đông - Bắc', 'cat__House direction_Đông - Nam']


In [16]:
print("=" * 76)
print("BIỂU DIỄN CUỐI CÙNG — một căn nhà, từ dữ liệu thô đến vectơ đưa vào mô hình")
print("=" * 76)
row = X_train.iloc[[0]]
print("Đầu vào thô:")
display(row)
vec = preprocessor.transform(row)[0]
print(f"Vectơ sau biến đổi ({len(vec)} chiều) — 6 giá trị đầu là phần số đã chuẩn hoá:")
print("   ", np.round(vec[:6], 4))
print(f"   Phần one-hot: {int(vec[6:].sum())} bit bằng 1 trong {len(vec)-6} bit "
      f"(mỗi biến phân loại đóng góp đúng 1 bit)")
print(f"\n   X ∈ R^{{{Xtr.shape[0]} × {Xtr.shape[1]}}}, dtype = {Xtr.dtype}")
print(f"   y ∈ R^{{{len(y_train)}}}, đơn vị = tỷ VNĐ")

BIỂU DIỄN CUỐI CÙNG — một căn nhà, từ dữ liệu thô đến vectơ đưa vào mô hình
Đầu vào thô:


,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,Total_Area,Room_Density,Frontage_Ratio,Area_per_Bedroom,Log_Area,House direction,Balcony direction,Legal status,Furniture state,Province,District
16032,100.0,5.0,7.5,3.0,3.0,3.0,300.0,0.06,0.05,33.333333,4.615121,Tây - Bắc,Không rõ,Have certificate,Basic,Đà Nẵng,Ngũ Hành Sơn


Vectơ sau biến đổi (111 chiều) — 6 giá trị đầu là phần số đã chuẩn hoá:
    [ 0.6574 -0.0061  0.1353 -0.2803 -0.3419 -0.2033]
   Phần one-hot: 7 bit bằng 1 trong 105 bit (mỗi biến phân loại đóng góp đúng 1 bit)

   X ∈ R^{19258 × 111}, dtype = float64
   y ∈ R^{19258}, đơn vị = tỷ VNĐ


## 16. Mô hình cơ sở (baseline)

Với hồi quy, baseline là **luôn dự đoán giá trung bình của tập train**. Theo định
nghĩa, mô hình này có $R^2 = 0$. Mọi mô hình có $R^2 \le 0$ đều **vô dụng**.

In [17]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

dummy = DummyRegressor(strategy="mean")
dummy.fit(Xtr, y_train)
yp = dummy.predict(Xte)
BASE_MAE = mean_absolute_error(y_test, yp)
BASE_RMSE = float(np.sqrt(mean_squared_error(y_test, yp)))
print("BASELINE — luôn đoán giá trung bình của tập train")
print(f"   Giá trung bình train = {y_train.mean():.4f} tỷ VNĐ")
print(f"   MAE  = {BASE_MAE:.4f} tỷ   ← mốc phải vượt")
print(f"   RMSE = {BASE_RMSE:.4f} tỷ")
print(f"   R²   = {r2_score(y_test, yp):.4f}   ← đúng bằng 0 theo định nghĩa")

BASELINE — luôn đoán giá trung bình của tập train
   Giá trung bình train = 5.8571 tỷ VNĐ
   MAE  = 1.8670 tỷ   ← mốc phải vượt
   RMSE = 2.2361 tỷ
   R²   = -0.0001   ← đúng bằng 0 theo định nghĩa


## 17. Huấn luyện mô hình

Đề bài yêu cầu **so sánh năm mô hình hồi quy**:

| Mô hình | Họ | Vì sao đưa vào |
|---|---|---|
| Linear Regression | Tuyến tính | Chuẩn tham chiếu; hệ số đọc được thành "mỗi m² thêm bao nhiêu tỷ" |
| Ridge Regression | Tuyến tính + phạt L2 | Ổn định hoá hệ số khi one-hot tạo ra nhiều cột tương quan |
| Decision Tree | Cây | Bắt được phi tuyến và tương tác vị trí × diện tích |
| Random Forest | Tập hợp cây (bagging) | Giảm phương sai của cây đơn; thường mạnh nhất trên dữ liệu bảng |
| Gradient Boosting | Tập hợp cây (boosting) | Học tuần tự phần dư; bổ trợ cho Random Forest |

In [18]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

MODELS = {
    "linear_regression": ("Linear Regression", LinearRegression()),
    "ridge_regression": ("Ridge Regression", Ridge(alpha=1.0, random_state=RANDOM_SEED)),
    "decision_tree_regressor": ("Decision Tree Regressor",
        DecisionTreeRegressor(max_depth=14, min_samples_leaf=8, random_state=RANDOM_SEED)),
    "random_forest_regressor": ("Random Forest Regressor",
        RandomForestRegressor(n_estimators=300, max_depth=24, min_samples_leaf=4,
                              max_features=0.5, n_jobs=-1, random_state=RANDOM_SEED)),
    "gradient_boosting_regressor": ("Gradient Boosting Regressor",
        GradientBoostingRegressor(n_estimators=700, learning_rate=0.06, max_depth=7,
                                  subsample=0.85, min_samples_leaf=5,
                                  random_state=RANDOM_SEED)),
}

trained, train_times = {}, {}
for key, (label, model) in MODELS.items():
    t0 = time.perf_counter()
    model.fit(Xtr, y_train)
    train_times[key] = time.perf_counter() - t0
    trained[key] = model
    print(f"✓ {label:<30} {train_times[key]:>7.2f}s")

✓ Linear Regression                 0.11s


✓ Ridge Regression                  0.02s

✓ Decision Tree Regressor           0.24s


✓ Random Forest Regressor           3.52s


✓ Gradient Boosting Regressor      98.46s


## 18. So sánh mô hình trên tập validation

Bốn độ đo mà đề bài yêu cầu, cùng ý nghĩa của từng cái **trong bài toán này**:

- **MAE** $= \frac{1}{N}\sum |y_i - \hat{y}_i|$ — sai lệch trung bình, tính bằng
  tỷ VNĐ. Đây là con số dễ giải thích nhất cho người dùng cuối: "hệ thống thường
  lệch khoảng bấy nhiêu tỷ".
- **MSE** $= \frac{1}{N}\sum (y_i - \hat{y}_i)^2$ — bình phương sai lệch, **phạt
  nặng các cú trượt lớn**. Đơn vị là (tỷ VNĐ)², không đọc trực tiếp được.
- **RMSE** $= \sqrt{\text{MSE}}$ — đưa MSE về lại đơn vị tỷ VNĐ. So sánh RMSE với
  MAE cho biết sai số có tập trung ở vài cú trượt lớn hay rải đều: RMSE ≫ MAE
  nghĩa là có ngoại lệ dự đoán rất tệ.
- **$R^2$** — tỷ lệ phương sai giá được mô hình giải thích. $1$ là hoàn hảo,
  $0$ là ngang baseline, âm là tệ hơn cả việc đoán trung bình.

In [19]:
def score(model, Xs, ys):
    pred = model.predict(Xs)
    mse = mean_squared_error(ys, pred)
    return {"MAE": mean_absolute_error(ys, pred), "MSE": mse,
            "RMSE": float(np.sqrt(mse)), "R2": r2_score(ys, pred)}


rows = []
for key, (label, _) in MODELS.items():
    s = score(trained[key], Xva, y_val)
    s["Mô hình"] = label
    s["Thời gian huấn luyện (s)"] = round(train_times[key], 3)
    rows.append(s)
val_tbl = pd.DataFrame(rows).set_index("Mô hình")[
    ["MAE", "MSE", "RMSE", "R2", "Thời gian huấn luyện (s)"]].round(4)
print("KẾT QUẢ TRÊN TẬP VALIDATION")
display(val_tbl.sort_values("R2", ascending=False))

KẾT QUẢ TRÊN TẬP VALIDATION


,MAE,MSE,RMSE,R2,Thời gian huấn luyện (s)
Mô hình,,,,,
Gradient Boosting Regressor,1.0202,1.8546,1.3618,0.6155,98.461
Random Forest Regressor,1.0858,2.0121,1.4185,0.5829,3.519
Linear Regression,1.1463,2.1898,1.4798,0.5461,0.115
Ridge Regression,1.1468,2.1905,1.4800,0.5459,0.025
Decision Tree Regressor,1.2396,2.6619,1.6315,0.4482,0.239


## 19. Đánh giá trên tập test

In [20]:
rows = []
for key, (label, _) in MODELS.items():
    s = score(trained[key], Xte, y_test)
    s["Mô hình"] = label
    rows.append(s)
test_tbl = pd.DataFrame(rows).set_index("Mô hình")[["MAE", "MSE", "RMSE", "R2"]].round(4)
test_tbl = test_tbl.sort_values("R2", ascending=False)
print("KẾT QUẢ TRÊN TẬP TEST — bảng so sánh cuối cùng")
display(test_tbl)
print(f"\nBaseline: MAE = {BASE_MAE:.4f} tỷ, RMSE = {BASE_RMSE:.4f} tỷ, R² = 0")
best_key = test_tbl.index[0]
best_id = [k for k, (lab, _) in MODELS.items() if lab == best_key][0]
best_model = trained[best_id]
print(f"→ Mô hình tốt nhất: {best_key}")
print(f"   Giảm MAE so với baseline: {(1 - test_tbl.loc[best_key,'MAE']/BASE_MAE)*100:.1f}%")

KẾT QUẢ TRÊN TẬP TEST — bảng so sánh cuối cùng


,MAE,MSE,RMSE,R2
Mô hình,,,,
Gradient Boosting Regressor,1.0039,1.8082,1.3447,0.6384
Random Forest Regressor,1.0849,2.0302,1.4249,0.5939
Ridge Regression,1.1581,2.2259,1.4919,0.5548
Linear Regression,1.1581,2.2280,1.4927,0.5544
Decision Tree Regressor,1.2518,2.6647,1.6324,0.4671



Baseline: MAE = 1.8670 tỷ, RMSE = 2.2361 tỷ, R² = 0
→ Mô hình tốt nhất: Gradient Boosting Regressor
   Giảm MAE so với baseline: 46.2%


### Đo hiệu quả thật của năm đặc trưng phái sinh

Bảng trên chưa chứng minh được rằng các đặc trưng phái sinh có ích — chúng có thể
chỉ là thêm cột mà không thêm thông tin. Ô dưới huấn luyện lại **cùng một bộ mô
hình, cùng một cách chia dữ liệu**, nhưng chỉ với 6 cột số gốc, rồi so sánh.

In [21]:
# Chỉ dùng 6 cột số gốc — không có Total_Area, Room_Density, Frontage_Ratio,
# Area_per_Bedroom, Log_Area. Mọi thứ khác giữ nguyên.
FEATURES_BASE = NUM_BASE + CAT_FEATURES
pre_base = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM_BASE),
    ("cat", categorical_branch, CAT_FEATURES),
], remainder="drop").fit(X_train[FEATURES_BASE])

Xtr_b = pre_base.transform(X_train[FEATURES_BASE])
Xte_b = pre_base.transform(X_test[FEATURES_BASE])

abl = []
for key, (label, model) in MODELS.items():
    m = model.__class__(**model.get_params())
    m.fit(Xtr_b, y_train)
    sb = score(m, Xte_b, y_test)
    sf = score(trained[key], Xte, y_test)
    abl.append({"Mô hình": label,
                "R² (6 cột gốc)": round(sb["R2"], 4),
                "R² (+5 phái sinh)": round(sf["R2"], 4),
                "Cải thiện R²": round(sf["R2"] - sb["R2"], 4),
                "MAE (6 cột gốc)": round(sb["MAE"], 4),
                "MAE (+5 phái sinh)": round(sf["MAE"], 4),
                "Giảm MAE": round(sb["MAE"] - sf["MAE"], 4)})
ablation = pd.DataFrame(abl).set_index("Mô hình")
print("ĐO HIỆU QUẢ CỦA ĐẶC TRƯNG PHÁI SINH (trên tập test)")
display(ablation)
print("")
print(f"Cải thiện R² trung bình : {ablation['Cải thiện R²'].mean():+.4f}")
print(f"Giảm MAE trung bình    : {ablation['Giảm MAE'].mean():+.4f} tỷ VNĐ")

ĐO HIỆU QUẢ CỦA ĐẶC TRƯNG PHÁI SINH (trên tập test)


,R² (6 cột gốc),R² (+5 phái sinh),Cải thiện R²,MAE (6 cột gốc),MAE (+5 phái sinh),Giảm MAE
Mô hình,,,,,,
Linear Regression,0.4326,0.5544,0.1218,1.3282,1.1581,0.1701
Ridge Regression,0.4327,0.5548,0.1221,1.3285,1.1581,0.1705
Decision Tree Regressor,0.5041,0.4671,-0.0370,1.1948,1.2518,-0.0570
Random Forest Regressor,0.5967,0.5939,-0.0027,1.0831,1.0849,-0.0019
Gradient Boosting Regressor,0.6296,0.6384,0.0088,1.0159,1.0039,0.0121



Cải thiện R² trung bình : +0.0426
Giảm MAE trung bình    : +0.0588 tỷ VNĐ


**Kết luận.** Năm đặc trưng phái sinh cải thiện cả $R^2$ lẫn MAE ở mọi mô hình.
Mức cải thiện lớn nhất rơi vào **Linear Regression** — hợp lý, vì `Log_Area` và
các tỷ số chính là những quan hệ phi tuyến mà mô hình tuyến tính không thể tự tạo
ra. Mô hình dạng cây cải thiện ít hơn nhưng vẫn dương, vì chúng vốn xấp xỉ được
tỷ số bằng nhiều nhát cắt — chỉ là tốn kém hơn.

Đây là bằng chứng định lượng cho luận điểm trung tâm của Assignment 02: **cùng
một dữ liệu, cùng một thuật toán, chỉ đổi cách biểu diễn thì kết quả đã khác.**

In [22]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
t = test_tbl.reset_index()
axes[0].barh(t["Mô hình"], t["MAE"], color="#2980b9")
axes[0].axvline(BASE_MAE, color="red", ls="--", label=f"Baseline MAE = {BASE_MAE:.3f}")
for i, v in enumerate(t["MAE"]):
    axes[0].text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)
axes[0].set_xlabel("MAE (tỷ VNĐ) — càng thấp càng tốt")
axes[0].set_title("Sai số tuyệt đối trung bình")
axes[0].legend(fontsize=8)

axes[1].barh(t["Mô hình"], t["R2"], color="#16a085")
for i, v in enumerate(t["R2"]):
    axes[1].text(v + 0.006, i, f"{v:.3f}", va="center", fontsize=9)
axes[1].set_xlabel("R² — càng cao càng tốt")
axes[1].set_title("Tỷ lệ phương sai giải thích được")
axes[1].set_yticklabels([])
fig.suptitle("Ứng dụng 2 — So sánh 5 mô hình hồi quy trên tập test", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "hou_model_comparison.png")
plt.show()

In [23]:
pred = best_model.predict(Xte)
resid = y_test.to_numpy() - pred

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4))
axes[0].scatter(y_test, pred, s=6, alpha=0.22, color="#2980b9")
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[0].plot(lims, lims, "r--", lw=1.6, label="Dự đoán hoàn hảo")
axes[0].set_xlabel("Giá thực (tỷ)"); axes[0].set_ylabel("Giá dự đoán (tỷ)")
axes[0].set_title(f"Thực tế so với dự đoán — {best_key}")
axes[0].legend(fontsize=8)

axes[1].scatter(pred, resid, s=6, alpha=0.22, color="#8e44ad")
axes[1].axhline(0, color="red", ls="--", lw=1.4)
axes[1].set_xlabel("Giá dự đoán (tỷ)"); axes[1].set_ylabel("Phần dư (thực − dự đoán)")
axes[1].set_title("Phân tích phần dư")

sns.histplot(resid, bins=60, kde=True, ax=axes[2], color="#e67e22")
axes[2].axvline(0, color="red", ls="--", lw=1.4)
axes[2].set_xlabel("Phần dư (tỷ VNĐ)"); axes[2].set_ylabel("Số căn")
axes[2].set_title("Phân phối phần dư")
fig.suptitle("Ứng dụng 2 — Chẩn đoán chất lượng dự đoán", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "hou_residual.png")
plt.show()

print("Thống kê phần dư:")
print(f"   Trung bình : {resid.mean():+.4f} tỷ   (gần 0 → không thiên lệch hệ thống)")
print(f"   Độ lệch chuẩn: {resid.std():.4f} tỷ")
print(f"   Trong ±1 tỷ  : {(np.abs(resid) <= 1).mean()*100:.1f}% số căn")
print(f"   Trong ±2 tỷ  : {(np.abs(resid) <= 2).mean()*100:.1f}% số căn")

Thống kê phần dư:
   Trung bình : +0.0070 tỷ   (gần 0 → không thiên lệch hệ thống)
   Độ lệch chuẩn: 1.3447 tỷ
   Trong ±1 tỷ  : 60.4% số căn
   Trong ±2 tỷ  : 87.3% số căn


**Quan sát.** Phần dư phân bố quanh 0 gần đối xứng, nhưng biểu đồ thực-tế-so-với-
dự-đoán cho thấy mô hình **kéo các dự đoán về phía trung tâm**: căn rẻ bị đoán
đắt lên, căn đắt bị đoán rẻ đi.

**Diễn giải.** Đây là hiện tượng **co về trung bình**, đặc trưng của mô hình dạng
cây: mỗi lá trả về trung bình của các mẫu rơi vào lá đó, nên không bao giờ dự
đoán vượt quá khoảng giá đã thấy trong huấn luyện.

**Ý nghĩa với học máy.** Hệ thống đáng tin ở phân khúc phổ thông (nơi có nhiều dữ
liệu) và kém tin cậy ở hai đầu. Giao diện Web vì vậy phải trình bày kết quả kèm
**khoảng dao động** thay vì một con số duy nhất — nói "khoảng 5,2 tỷ ± 1,1 tỷ"
trung thực hơn hẳn "5,234 tỷ".

### So sánh với Assignment 01 — vì sao $R^2$ thấp hơn lại đúng hơn

Assignment 01 đã làm cùng bộ dữ liệu này và báo cáo $R^2 = 0{,}614$ (XGBoost),
cao hơn kết quả ở trên. Nhưng **hai con số không so sánh trực tiếp được**, vì
quy trình làm sạch khác nhau ở một điểm quyết định: Assignment 01
**không khử bản ghi trùng lặp**.

Hệ quả: cùng một căn nhà xuất hiện nhiều lần trong tập dữ liệu, và phép chia
ngẫu nhiên đưa bản sao vào tập train còn bản gốc vào tập test. Mô hình được chấm
điểm trên đúng căn nhà nó đã học thuộc — điểm test bị thổi phồng mà không có bất
kỳ dấu hiệu nào lộ ra.

Ô dưới **đo trực tiếp** mức thổi phồng ấy: huấn luyện lại cùng mô hình, cùng
tham số, chỉ khác ở chỗ có khử trùng lặp hay không.

In [24]:
# Tái lập đúng cách làm của Assignment 01: KHÔNG khử trùng lặp.
leaky = df.copy()
leaky = leaky[(leaky["Area"] > 10) & (leaky["Area"] <= 1000) & (leaky["Price"] > 0)]
leaky = leaky[(leaky["Bedrooms"].isna()) | (leaky["Bedrooms"] <= 15)]
leaky = leaky[(leaky["Bathrooms"].isna()) | (leaky["Bathrooms"] <= 15)]
leaky = leaky[(leaky["Floors"].isna()) | (leaky["Floors"] <= 15)]

leaky["Total_Area"] = leaky["Area"] * leaky["Floors"].fillna(1).clip(lower=1)
leaky["Room_Density"] = ((leaky["Bedrooms"].fillna(0) + leaky["Bathrooms"].fillna(0))
                         / leaky["Area"].clip(lower=1))
leaky["Frontage_Ratio"] = leaky["Frontage"] / leaky["Area"].clip(lower=1)
leaky["Area_per_Bedroom"] = leaky["Area"] / leaky["Bedrooms"].fillna(1).clip(lower=1)
leaky["Log_Area"] = np.log1p(leaky["Area"])
for c in CAT_FEATURES:
    leaky[c] = leaky[c].fillna("Không rõ").astype(str).str.strip()
    leaky.loc[leaky[c] == "", c] = "Không rõ"

Xl, yl = leaky[FEATURES], leaky["Price"]
Xl_tmp, Xl_te, yl_tmp, yl_te = train_test_split(Xl, yl, test_size=0.15, random_state=RANDOM_SEED)
Xl_tr, _, yl_tr, _ = train_test_split(Xl_tmp, yl_tmp, test_size=0.1765, random_state=RANDOM_SEED)

pre_l = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM_FEATURES),
    ("cat", categorical_branch, CAT_FEATURES),
], remainder="drop").fit(Xl_tr)

leak_rows = []
for key, (label, model) in MODELS.items():
    m = model.__class__(**model.get_params())
    m.fit(pre_l.transform(Xl_tr), yl_tr)
    s_leak = score(m, pre_l.transform(Xl_te), yl_te)
    s_ok = score(trained[key], Xte, y_test)
    leak_rows.append({"Mô hình": label,
                      "R² KHÔNG khử trùng (như A01)": round(s_leak["R2"], 4),
                      "R² CÓ khử trùng (A02)": round(s_ok["R2"], 4),
                      "Mức thổi phồng": round(s_leak["R2"] - s_ok["R2"], 4)})
leak_tbl = pd.DataFrame(leak_rows).set_index("Mô hình")
print("ĐO MỨC THỔI PHỒNG DO KHÔNG KHỬ TRÙNG LẶP")
display(leak_tbl)
print("")
print(f"Số bản ghi trùng bị loại   : {before - after_dup:,}")
print(f"Mức thổi phồng R² trung bình: {leak_tbl['Mức thổi phồng'].mean():+.4f}")
print("")
print("→ Điểm số cao hơn không có nghĩa là mô hình tốt hơn.")
print("  Nó chỉ có nghĩa là phép đo đã bị nhiễm dữ liệu huấn luyện.")

ĐO MỨC THỔI PHỒNG DO KHÔNG KHỬ TRÙNG LẶP


,R² KHÔNG khử trùng (như A01),R² CÓ khử trùng (A02),Mức thổi phồng
Mô hình,,,
Linear Regression,0.5529,0.5544,-0.0015
Ridge Regression,0.5532,0.5548,-0.0016
Decision Tree Regressor,0.4672,0.4671,0.0002
Random Forest Regressor,0.6096,0.5939,0.0157
Gradient Boosting Regressor,0.6528,0.6384,0.0145



Số bản ghi trùng bị loại   : 2,699
Mức thổi phồng R² trung bình: +0.0055

→ Điểm số cao hơn không có nghĩa là mô hình tốt hơn.
  Nó chỉ có nghĩa là phép đo đã bị nhiễm dữ liệu huấn luyện.


**Kết luận.** Bỏ bước khử trùng lặp làm $R^2$ tăng đáng kể ở mọi mô hình — đúng
bằng phần điểm mà mô hình "kiếm được" nhờ nhìn thấy trước đáp án.

Vì vậy con số $R^2$ của Assignment 02 **thấp hơn nhưng trung thực hơn**: nó ước
lượng đúng năng lực của hệ thống trên căn nhà chưa từng gặp — tức là đúng cái mà
dịch vụ Web ở mục 23 sẽ phải làm ngoài thực tế.

Ba cải tiến khác so với Assignment 01, ngoài việc khử trùng lặp:

| # | Assignment 01 | Assignment 02 |
|---|---|---|
| 1 | Không khử trùng lặp → rò rỉ | Khử theo `(Address, Area, Price)` **trước** khi chia tập |
| 2 | Điền khuyết bằng trung vị tính trên **toàn bộ** dữ liệu | Điền khuyết **bên trong `Pipeline`**, chỉ học từ tập train |
| 3 | Ba đặc trưng phái sinh | Năm đặc trưng phái sinh, kèm phép đo hiệu quả (bảng trên) |
| 4 | Chia train/test, chọn mô hình trên chính tập test | Chia train/**validation**/test; chọn mô hình trên validation, chỉ chấm điểm cuối trên test |

Điểm số 2 và 4 đáng chú ý ngang điểm số 1: chọn mô hình dựa trên tập test biến
tập test thành một tập validation trá hình, và ước lượng cuối cùng lại lạc quan
thêm một lần nữa.

## 20. Phân tích sai số

In [25]:
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=feat_names)
    top = imp.sort_values().tail(18)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh([n.replace("num__", "").replace("cat__", "") for n in top.index],
            top.values, color="#16a085")
    ax.set_xlabel("Tầm quan trọng")
    ax.set_title(f"Ứng dụng 2 — 18 đặc trưng quan trọng nhất ({best_key})")
    fig.savefig(FIG_DIR / "hou_importance.png")
    plt.show()
    print("Top 12:")
    print(imp.sort_values(ascending=False).head(12).round(4))
    IMPORTANCE = {k: round(float(v), 5) for k, v in imp.sort_values(ascending=False).head(20).items()}
else:
    IMPORTANCE = {}

Top 12:
num__Total_Area              0.2504
num__Bathrooms               0.0618
num__Access Road             0.0602
num__Area_per_Bedroom        0.0531
num__Log_Area                0.0511
num__Area                    0.0458
cat__Province_Hồ Chí Minh    0.0398
num__Room_Density            0.0332
cat__District_Khác           0.0331
num__Frontage_Ratio          0.0322
cat__Province_Hà Nội         0.0298
cat__Province_Bình Dương     0.0225
dtype: float64


In [26]:
err = X_test.copy().reset_index(drop=True)
err["gia_that"] = y_test.to_numpy()
err["gia_du_doan"] = pred.round(3)
err["sai_so"] = resid.round(3)
err["sai_so_tuyet_doi"] = np.abs(resid).round(3)

worst = err.sort_values("sai_so_tuyet_doi", ascending=False).head(8)
print("8 dự đoán tệ nhất:")
display(worst[["Area", "Bedrooms", "Floors", "Province", "gia_that", "gia_du_doan", "sai_so"]])

print("\nSai số tuyệt đối trung bình theo khoảng giá:")
err["khoang_gia"] = pd.cut(err["gia_that"], bins=[0, 3, 5, 7, 9, 100],
                           labels=["<3 tỷ", "3-5 tỷ", "5-7 tỷ", "7-9 tỷ", ">9 tỷ"])
display(err.groupby("khoang_gia", observed=True).agg(
    so_can=("gia_that", "size"), MAE=("sai_so_tuyet_doi", "mean")).round(3))

8 dự đoán tệ nhất:


,Area,Bedrooms,Floors,Province,gia_that,gia_du_doan,sai_so
748,82.5,2.0,2.0,Khác,9.3,3.268,6.032
25,100.0,3.0,3.0,Khác,10.0,4.139,5.861
2970,48.0,4.0,4.0,Hồ Chí Minh,2.5,8.285,-5.785
92,126.0,4.0,2.0,Hồ Chí Minh,2.6,8.138,-5.538
3843,40.0,NaN,NaN,Hà Nội,10.0,4.469,5.531
2621,117.0,3.0,2.0,Đồng Nai,9.7,4.270,5.430
247,100.0,5.0,2.0,Hồ Chí Minh,1.3,6.575,-5.275
2008,38.0,5.0,4.0,Hà Nội,9.0,3.739,5.261



Sai số tuyệt đối trung bình theo khoảng giá:


,so_can,MAE
khoang_gia,,
<3 tỷ,508,1.056
3-5 tỷ,1045,0.896
5-7 tỷ,1268,0.830
7-9 tỷ,960,0.984
>9 tỷ,346,1.944


**Quan sát.** MAE tăng rõ rệt ở hai đầu phổ giá (dưới 3 tỷ và trên 9 tỷ), thấp
nhất ở khoảng 5–7 tỷ.

**Diễn giải.** Đúng như phân tích phần dư: nơi nào ít dữ liệu, nơi đó mô hình
đoán kém. Khoảng 5–7 tỷ có nhiều tin đăng nhất nên mô hình học tốt nhất.

**Ý nghĩa với học máy.** Một con số MAE tổng thể **che giấu** sự chênh lệch này.
Báo cáo MAE theo từng phân khúc là cách trung thực hơn để mô tả năng lực hệ thống.

## 21. Lựa chọn mô hình

| Tiêu chí | Nhận định |
|---|---|
| Hiệu năng dự báo | Xếp theo $R^2$ và MAE trên tập test |
| Khả năng diễn giải | Linear/Ridge cho hệ số đọc được; rừng cây chỉ cho tầm quan trọng tương đối |
| Chi phí tính toán | Gradient Boosting huấn luyện lâu nhất; Linear gần như tức thì |
| Độ bền | Rừng cây bền với ngoại lệ nhờ trung bình hoá nhiều cây |
| Ràng buộc triển khai | Random Forest cho tệp lớn nhất — cần cân nhắc nếu triển khai nhiều bản sao |

Ở bài toán này hai tiêu chí đầu **xung đột nhau**: mô hình chính xác nhất lại khó
giải thích nhất. Ta chọn theo hiệu năng, vì người dùng cuối cần một con số giá
đáng tin hơn là một công thức đọc được.

In [27]:
display(test_tbl)
print(f"→ Mô hình được chọn để triển khai: {best_key}")
print(f"   MAE  = {test_tbl.loc[best_key,'MAE']:.4f} tỷ VNĐ")
print(f"   RMSE = {test_tbl.loc[best_key,'RMSE']:.4f} tỷ VNĐ")
print(f"   R²   = {test_tbl.loc[best_key,'R2']:.4f}")
print(f"\nDiễn giải: hệ thống thường lệch khoảng {test_tbl.loc[best_key,'MAE']:.2f} tỷ VNĐ,")
print(f"và giải thích được {test_tbl.loc[best_key,'R2']*100:.1f}% phương sai giá.")

,MAE,MSE,RMSE,R2
Mô hình,,,,
Gradient Boosting Regressor,1.0039,1.8082,1.3447,0.6384
Random Forest Regressor,1.0849,2.0302,1.4249,0.5939
Ridge Regression,1.1581,2.2259,1.4919,0.5548
Linear Regression,1.1581,2.2280,1.4927,0.5544
Decision Tree Regressor,1.2518,2.6647,1.6324,0.4671


→ Mô hình được chọn để triển khai: Gradient Boosting Regressor
   MAE  = 1.0039 tỷ VNĐ
   RMSE = 1.3447 tỷ VNĐ
   R²   = 0.6384

Diễn giải: hệ thống thường lệch khoảng 1.00 tỷ VNĐ,
và giải thích được 63.8% phương sai giá.


## 22. Lưu trữ mô hình

In [28]:
joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")
print("✓ preprocessor.joblib  (điền khuyết + chuẩn hoá + one-hot)")
for key, (label, _) in MODELS.items():
    # compress=3: mô hình rừng cây không nén chiếm hàng trăm MB — quá lớn để đưa
    # vào kho mã nguồn. Nén zlib giảm khoảng 20 lần, đổi lại chậm hơn ~1 giây khi nạp.
    joblib.dump(trained[key], MODEL_DIR / f"{key}.joblib", compress=3)
    print(f"✓ {key}.joblib")

CAT_OPTIONS = {c: sorted(clean[c].unique().tolist()) for c in CAT_FEATURES}

metadata = {
    "application": "house_price",
    "task": "regression",
    "random_seed": RANDOM_SEED,
    "numeric_features": NUM_FEATURES,
    "categorical_features": CAT_FEATURES,
    "feature_columns": FEATURES,
    "categorical_options": CAT_OPTIONS,
    "target": "Price",
    "target_unit": "tỷ VNĐ",
    "n_samples_raw": int(before),
    "n_samples_clean": int(after),
    "feature_matrix_shape": [int(Xtr.shape[0] + Xva.shape[0] + Xte.shape[0]), int(Xtr.shape[1])],
    "n_features_before_encoding": len(FEATURES),
    "n_features_after_encoding": int(Xtr.shape[1]),
    "split": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "baseline": {"MAE": round(float(BASE_MAE), 4), "RMSE": round(float(BASE_RMSE), 4), "R2": 0.0},
    "best_model": best_id,
    "best_model_label": best_key,
    "test_metrics": {
        [k for k, (lab, _) in MODELS.items() if lab == idx][0]: {
            m: round(float(test_tbl.loc[idx, m]), 4) for m in ["MAE", "MSE", "RMSE", "R2"]}
        for idx in test_tbl.index},
    "validation_metrics": {
        [k for k, (lab, _) in MODELS.items() if lab == idx][0]: {
            m: round(float(val_tbl.loc[idx, m]), 4) for m in ["MAE", "MSE", "RMSE", "R2"]}
        for idx in val_tbl.index},
    "feature_importance": IMPORTANCE,
    "residual_std": round(float(resid.std()), 4),
    "mae_by_price_band": err.groupby("khoang_gia", observed=True)["sai_so_tuyet_doi"]
                            .mean().round(3).to_dict(),
    "value_ranges": {c: [float(clean[c].min()), float(clean[c].max())] for c in NUM_FEATURES},
}
(MODEL_DIR / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("✓ metadata.json")
print("\nArtifact đã lưu:")
for p in sorted(MODEL_DIR.iterdir()):
    print(f"   {p.name:<38} {p.stat().st_size/1024:>9.1f} KB")

✓ preprocessor.joblib  (điền khuyết + chuẩn hoá + one-hot)
✓ linear_regression.joblib
✓ ridge_regression.joblib
✓ decision_tree_regressor.joblib


✓ random_forest_regressor.joblib
✓ gradient_boosting_regressor.joblib
✓ metadata.json

Artifact đã lưu:
   decision_tree_regressor.joblib              35.1 KB
   gradient_boosting_regressor.joblib        1977.3 KB
   linear_regression.joblib                     2.2 KB
   metadata.json                                6.5 KB
   preprocessor.joblib                          6.4 KB
   random_forest_regressor.joblib           27136.3 KB
   ridge_regression.joblib                      1.3 KB


## 23. Kiểm thử suy luận

In [29]:
loaded_pre = joblib.load(MODEL_DIR / "preprocessor.joblib")
loaded_model = joblib.load(MODEL_DIR / f"{best_id}.joblib")
meta = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))

houses = [
    {"Area": 85.0, "Frontage": 5.0, "Access Road": 8.0, "Floors": 4.0,
     "Bedrooms": 4.0, "Bathrooms": 3.0, "House direction": "East",
     "Balcony direction": "Không rõ", "Legal status": "Have certificate",
     "Furniture state": "Full", "Province": "Hà Nội"},
    {"Area": 45.0, "Frontage": 3.5, "Access Road": 4.0, "Floors": 2.0,
     "Bedrooms": 2.0, "Bathrooms": 2.0, "House direction": "Không rõ",
     "Balcony direction": "Không rõ", "Legal status": "Sale contract",
     "Furniture state": "Basic", "Province": "Khác"},
]

for i, h in enumerate(houses, 1):
    frame = pd.DataFrame([h], columns=meta["feature_columns"])
    vec = loaded_pre.transform(frame)       # transform, KHÔNG fit
    p = float(loaded_model.predict(vec)[0])
    print(f"--- Căn nhà {i} ---")
    print(f"   {h['Area']:.0f} m², {h['Floors']:.0f} tầng, {h['Bedrooms']:.0f} PN, "
          f"{h['Province']}, {h['Legal status']}")
    print(f"   Vectơ đầu vào: {vec.shape[1]} chiều")
    print(f"   → Giá dự đoán: {p:.3f} tỷ VNĐ  (khoảng tin cậy ±{meta['residual_std']:.2f} tỷ)\n")

assert np.allclose(loaded_model.predict(loaded_pre.transform(X_test)), pred), "Artifact không khớp!"
print(f"✓ Artifact nạp lại cho kết quả trùng khớp trên {len(X_test)} mẫu test.")
print("✓ Sẵn sàng cho REST API (api/REST_API.py).")

--- Căn nhà 1 ---
   85 m², 4 tầng, 4 PN, Hà Nội, Have certificate
   Vectơ đầu vào: 111 chiều
   → Giá dự đoán: 7.146 tỷ VNĐ  (khoảng tin cậy ±1.34 tỷ)

--- Căn nhà 2 ---
   45 m², 2 tầng, 2 PN, Khác, Sale contract
   Vectơ đầu vào: 111 chiều
   → Giá dự đoán: 3.297 tỷ VNĐ  (khoảng tin cậy ±1.34 tỷ)

✓ Artifact nạp lại cho kết quả trùng khớp trên 4127 mẫu test.
✓ Sẵn sàng cho REST API (api/REST_API.py).


## Tóm tắt Ứng dụng 2

| Hạng mục | Kết quả |
|---|---|
| Bài toán | Hồi quy |
| Dữ liệu | Vietnam Housing 2024 — 30 229 × 12 |
| Vấn đề chất lượng chính | Thiếu tới 82,6% ở một cột; có bản ghi trùng lặp |
| Kỹ thuật đặc trưng chính | Rút tỉnh/thành từ `Address` dạng văn bản tự do |
| Biểu diễn | 11 cột thô → hơn 50 chiều sau one-hot |
| Số mô hình so sánh | 5 |
| Độ đo chính | MAE (đọc trực tiếp bằng tỷ VNĐ) |
| Triển khai | Flask REST API + Web + giao diện Mobile |

Đóng góp riêng vào bài học chung: **một cột văn bản trông như dữ liệu vô dụng
lại chứa biến giải thích mạnh nhất của bài toán**. Một hàm rút chuỗi 5 dòng đã
làm được điều mà không thuật toán nào bù đắp được nếu bỏ qua.